In [ ]:
%load_ext autoreload
%autoreload 2
import json
import re
from collections import Counter, defaultdict
import altair as alt
import numpy as np
import altair as alt
alt.data_transformers.enable("vegafusion")
import pandas as pd
from datasets import Dataset, load_dataset
from word2number import w2n
import scipy.stats as stats
from scipy.stats import chi2_contingency
from function import *
female_color = "#a7d503"  # Greenish-yellow
male_color = "#1872f4"    # Blue

# Use Sinebow for the third category (it auto-generates colors)
 
color_scheme =  alt.Scale(scheme='sinebow')

In [ ]:
ds = load_dataset("cxyzhang/pmc_llmExtraction_v7")

In [ ]:
ds

In [ ]:
## For human evaluation 
# shuffled_dataset = dataset["train"].shuffle(seed=42)
# human_evaluation = shuffled_dataset.select(range(400))

# df_label_eval = human_evaluation.to_pandas()[["pmcid", "title", "combined_label_2"]]
# df_label_eval["pmcid"] = df_label_eval["pmcid"].astype(str)
# df_label_eval.to_csv("./human_evaluated_labels.csv")

In [ ]:
# Create a dictionary to store PMCIDs for each case
case_pmcid_map = defaultdict(set)

# Iterate through the dataset and populate the mapping
for case, pmcid in zip(ds["train"]["case"], ds["train"]["pmcid"]):
    case_pmcid_map[case].add(str(pmcid))  # Convert pmcid to string for consistency

# Select only cases that have multiple PMCIDs
multi_pmcid_cases = {case: pmcids for case, pmcids in case_pmcid_map.items() if len(pmcids) > 1}

# Count number of cases with multiple PMCIDs
print(f"Total cases with multiple PMCIDs: {len(multi_pmcid_cases)}")

# Remove cases that are empty dictionaries
cleaned_cases = {case: pmcids for case, pmcids in multi_pmcid_cases.items() if case.strip() != "{}"}
# Retain only one PMC ID per case
deduplicated_cases = {case: list(pmcids)[0] for case, pmcids in cleaned_cases.items()}

In [ ]:

# Create a dictionary to store PMCIDs for each case
case_pmcid_map = defaultdict(set)

# Populate the mapping while ignoring empty "{}" cases
for case, pmcid in zip(ds["train"]["case"], ds["train"]["pmcid"]):
    if case.strip() != "{}":  # Ignore empty cases
        case_pmcid_map[case].add(str(pmcid))  # Convert pmcid to string for consistency

        # Retain only one representative PMCID per case
deduplicated_cases = {case: list(pmcids)[0] for case, pmcids in case_pmcid_map.items()}

pmcids_to_keep = set(deduplicated_cases.values())  # Extract only the retained PMCIDs
# Keep only records where 'pmcid' is in pmcids_to_keep
dataset =  ds["train"].filter(lambda example: str(example["pmcid"]) in pmcids_to_keep)


In [ ]:
df = dataset.to_pandas()

* Add Mesh Terms

In [ ]:
# Convert mesh_dict keys to int by removing 'PMC' and casting
mesh_dict_cleaned = {str(k.replace('PMC', '')): v for k, v in mesh_dict.items()}

In [ ]:
len(set(df["pmcid"].tolist()).intersection(set(mesh_dict_cleaned.keys())))

In [ ]:
# Now you can directly map without needing .astype(str)
df['mesh_terms'] = df['pmcid'].map(mesh_dict_cleaned)

In [ ]:
df[df["mesh_terms"].notna()]

In [ ]:
with open("./pmcid_list.json", "w") as f:
    pmcid = df["pmcid"].to_list()
    json.dump(pmcid, f)

In [ ]:
# How many entities were in the LLM-extracted labels vs. NER concepts?

In [ ]:

df_expand = df.copy()
df_expand["topic"] = df_expand["combined_label_2"].apply(lambda x: x.split(",") if isinstance(x, str) else x)

df_expand = df_expand.explode("combined_label_2")

In [ ]:
# SPlit label distribution 
split_labels = df_expand["topic"].dropna()
all_labels = [label.strip() for sublist in split_labels for label in sublist if label]
split_label_counts = Counter(all_labels)
split_label_df = pd.DataFrame(split_label_counts.items(), columns=['label', 'count'])

In [ ]:
split_label_df["count"].describe().apply(lambda x: round(x, 2))

In [ ]:
import pandas as pd
import altair as alt
from collections import Counter

# Convert label counts to DataFrame
label_counts = Counter(split_label_df["label"])
label_df = pd.DataFrame(label_counts.items(), columns=['label', 'count'])

def plot_label_distribution(label_df, n=30, bottom=False):
    
    if bottom:
        bottom_n_labels = label_df.nsmallest(n, "count")  # Select least frequent labels
        n_labels = bottom_n_labels
        name = f'Bottom {n} Least Frequent Medical Conditions'  # FIXED: Removed comma
    else: 
        top_n_labels = label_df.nlargest(n, "count")  # Select most frequent labels
        n_labels = top_n_labels
        name = f'Top {n} Most Frequent Medical Conditions'

    # Create bar chart
    label_chart = alt.Chart(n_labels).mark_bar().encode(
        x=alt.X('count:Q', title="Counts"),
        y=alt.Y('label:N', sort='-x', title=name),
        tooltip=['label', 'count'],
    ).properties(
        width=600,
        height=800
    )
    
    return label_chart



In [ ]:
topics_with_at_least_100 = split_label_df[split_label_df["count"]>=100]["label"].tolist()

In [ ]:
split_label_df.sort_values("count", ascending=False)[:20]

In [ ]:
top_20_charts = plot_label_distribution(split_label_df)
top_20_charts.save("./graphs/overal_top20_conditions.pdf")
top_20_charts

# Load Publication Yr

In [ ]:
# sorted(yr_df["publication_year"].unique())

In [ ]:
yr_df = pd.read_json("./data/filtered_pub_yr.json", orient="index")
yr_df = yr_df.reset_index()

In [ ]:
yr_df.columns = ["pmcid", "publication_year"]

In [ ]:
yr_df["publication_year"] = yr_df["publication_year"].str.strip()
yr_df = yr_df[yr_df["publication_year"] != "2024"] # 2024 was removed from consideration due to low occurence

In [ ]:
yr_df["pmcid"] = yr_df["pmcid"].astype(str)


In [ ]:
yr_df["publication_year"].value_counts()

In [ ]:
yr_df["publication_year"].value_counts()

In [ ]:
df2 = df.merge(yr_df, on="pmcid")

In [ ]:
df2["publication_year"] = pd.to_numeric(df2["publication_year"], errors="coerce").round(0).astype("Int64")

In [ ]:
# Filter out unknown years (assuming missing years are NaN or 0)

df_filtered = df2[(df2["publication_year"].notna()) & (df2["publication_year"] != 2024)]

# Count the occurrences of each publication year
pub_yr_dic = dict(Counter(df_filtered["publication_year"]))

# Convert dictionary to DataFrame
yr_count_df = pd.DataFrame(list(pub_yr_dic.items()), columns=["Year", "Count"])

# Create an Altair bar chart
chart = alt.Chart(yr_count_df).mark_bar().encode(
    x=alt.X("Year:O", title="Publication Year", sort="ascending", axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("Count:Q", title="Number of Publications"),
    tooltip=["Year", "Count"]
).properties(
    title="Distribution of Publications by Year",
    width=600,
    height=400
)
 

chart.save("./graphs/publication year distribution.pdf")
chart

In [ ]:

# # Ensure 'publication_year' is integer and remove NaNs
# df2["publication_year"] = pd.to_numeric(df2["publication_year"], errors="coerce").fillna(0).astype(int)

# # Drop rows with missing labels or years
# df_filtered = df2.dropna(subset=["combined_label_2", "publication_year"])
# df_filtered = df2[(df2["publication_year"].notna()) & (df2["publication_year"] > 0)]


# # Split labels and create a list of (year, label) pairs
# label_counts = []
# for _, row in df_filtered.iterrows():
#     year = row["publication_year"]
#     labels = str(row["combined_label_2"]).split(", ")  
#     for label in labels:
#         label_counts.append((year, label.strip()))

# # Convert to DataFrame
# label_df = pd.DataFrame(label_counts, columns=["Year", "Label"])

# # Count occurrences of each label per year
# label_freq = label_df.groupby(["Year", "Label"]).size().reset_index(name="Count")

# # Identify the most frequent label per year
# most_frequent_labels = label_freq.loc[label_freq.groupby("Year")["Count"].idxmax()]

# # Create Altair bar chart
# chart = alt.Chart(most_frequent_labels).mark_bar().encode(
#     x=alt.X("Year:O", title="Publication Year", sort="ascending"),
#     y=alt.Y("Count:Q", title="Most Frequent Label Count"),
#     color=alt.Color("Label:N", title="Most Frequent Label"),  # Different color per label
#     tooltip=["Year", "Label", "Count"]
# ).properties(
   
#     width=800,
#     height=400
# )


# chart.save("./graphs/most_frequent_label_per_year.pdf")
# # Display the chart
# chart


In [ ]:
df2["publication_year"] = pd.to_numeric(df2["publication_year"], errors="coerce").round(0).astype("Int64")

In [ ]:
len(df2)

In [ ]:
# import pandas as pd
# import altair as alt

# # Ensure 'publication_year' is integer and remove NaNs
# df2["publication_year"] = pd.to_numeric(df2["publication_year"], errors="coerce").fillna(0).astype(int)

# # Drop rows with missing labels or years
# df_filtered = df2.dropna(subset=["combined_label_2", "publication_year"])
# df_filtered = df2[(df2["publication_year"].notna()) & (df2["publication_year"] >= 2014) & (df2["publication_year"] < 2024)]

# # Split labels and create a list of (year, label) pairs
# label_counts = []
# for _, row in df_filtered.iterrows():
#     year = row["publication_year"]
#     labels = str(row["combined_label_2"]).split(", ")  
#     for label in labels:
#         label_counts.append((year, label.strip()))

# # Convert to DataFrame
# label_df = pd.DataFrame(label_counts, columns=["Year", "Label"])

# # Count occurrences of each label per year
# label_freq = label_df.groupby(["Year", "Label"]).size().reset_index(name="Count")

# # Get top 5 labels per year
# label_freq["Rank"] = label_freq.groupby("Year")["Count"].rank(method="first", ascending=False)
# top_labels_per_year = label_freq[label_freq["Rank"] <= 3]

# color_scheme = [
#     "#648FFF", "#785EF0", "#DC267F", "#FE6100", "#FFB000",  # IBM Design
#     "#4E79A7", "#F28E2B", "#E15759", "#76B7B2", "#59A14F",  # Tableau
#     "#E69F00", "#56B4E9", "#009E73", "#F0E442"  # Okabe-Ito
# ]


# # Create stacked bar chart
# chart = alt.Chart(top_labels_per_year).mark_bar().encode(
#     x=alt.X("Year:O", title="Publication Year", sort="ascending"),
#     y=alt.Y("Count:Q", title="Occurrences of Top 3 Topics", stack="zero"),
#     color=alt.Color("Label:N",
#                     scale=alt.Scale(domain=top_labels_per_year["Label"].unique(), range=color_scheme), legend=None),
#     tooltip=["Year", "Label", "Count"]
# ).properties(
#     width=1000,
#     height=500,
#     # title="Top 5 Most Frequent Labels Per Year"
# )

# # Save chart
# # chart.save("./graphs/Top 3 Most Frequent Topics in Case Reports Per Year.pdf")

# # Display the chart
# chart


#### Age 

In [ ]:
def extract_age(strings):
 
    results = []

    # Regex pattern to extract age with units (e.g., "26-year-old", "2 months old")
    age_pattern = re.compile(
        r'(\b(?:\d+|(?:one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|thirteen|fourteen|fifteen|sixteen|seventeen|eighteen|nineteen|twenty|thirty|forty|fifty|sixty|seventy|eighty|ninety)(?:[-\s]?(?:one|two|three|four|five|six|seven|eight|nine))?)\b)[-\s]*(year[s]?|yr[s]?|month[s]?|mon|y[s]?|wk[s]?|w[s]?|week[s]?|d[s]?|day[s]?)[-\s]*(old|of age)?', 
        re.IGNORECASE
    )


    for text in strings:
        # Search for age information
        match_age = age_pattern.search(text)
        if match_age:
            age_text = match_age.group(1)  # Extract the numeric or word-based part
            age_unit = match_age.group(2).lower()  # Extract the unit part
            
            # Convert word-based numbers to integer if necessary
            try:
                age_value = int(age_text) if age_text.isdigit() else w2n.word_to_num(age_text)
            except ValueError:
                age_value = None
            
            # Convert to years if age is in months, weeks, or days
            if age_value is not None:
                if 'month' in age_unit or 'mon' in age_unit:
                    age_in_years = round(age_value / 12, 2)
                    age = f"{age_in_years} years"
                elif 'week' in age_unit:
                    age_in_years = round(age_value / 52, 2)
                    age = f"{age_in_years} years"
                elif 'day' in age_unit:
                    age_in_years = round(age_value / 365, 2)
                    age = f"{age_in_years} years"
                else:
                    # If the age is already in years
                    age = f"{age_value} years"
            else:
                age = "unspecified"
        else:
            age = "unspecified"

        # Store the extracted age
        results.append({'Age': age})
    
    return results

In [ ]:
from word2number import w2n
age = extract_age(dataset["case"])
age_dict = {case: i["Age"] for case, i in zip(dataset["pmcid"], age)}
age_df = pd.DataFrame(list(age_dict.items()), columns=["pmcid", "Age"])

In [ ]:
len({case: i["Age"] for case, i in zip(dataset["pmcid"], age)if i["Age"]=="unspecified"})/len(dataset["case"])

In [ ]:
pmcid_counts = Counter(dataset["pmcid"])
duplicates = {k: v for k, v in pmcid_counts.items() if v > 1}
print(f"Total duplicate pmcids: {len(duplicates)}")

The same clinical case might be discussed in different articles, leading to multiple PMCIDs per case.

In [ ]:
age_df["Age (years)"] = age_df["Age"].str.extract(r'(\d+\.?\d*)')  # Now captures decimals too
# Convert only valid numbers to float, leave NaN as is
age_df["Age (years)"] = pd.to_numeric(age_df["Age (years)"], errors='coerce')

bins = [0, 1/12, 1.5, 11, 16, 41, 65, float('inf')]  
labels = [
    'Neonatal',    # 0 - 1 month
    'Infancy',     # >1 month - 1.5 years
    'Childhood',   # >1.5 - 11 years
    'Adolescence', # >11 - 16 years
    'Adulthood (16-41 yr)',  # >16 - 41 years
    'Adulthood (41-64 yr)',  # >41 - 65 years
    'Adulthood (>64 yr)'     # >65 years
]


# Bin numeric ages
age_df["Age Category"] = pd.cut(pd.to_numeric(age_df["Age (years)"], errors='coerce'), bins=bins, labels=labels, right=True)
# Convert all NaN values in "Age Category" to "Unspecified"
age_df["Age Category"] = age_df["Age Category"].astype(str).replace({"nan": "Unspecified", np.nan: "Unspecified"})

# Convert to string type before assigning "Unspecified"
age_df["Age Category"] = age_df["Age Category"].astype(str)

# Assign "Unspecified" for rows where Age was missing
age_df.loc[age_df["Age"].str.lower().str.contains("unspecified", na=False), "Age Category"] = "Unspecified"



In [ ]:
print(age_df[age_df["Age (years)"].isna()]["Age"].unique())
print(age_df["Age Category"].unique())  

In [ ]:
 
filtered_df = age_df[age_df["Age Category"].notna()]

# Compute total number of cases (excluding "Unspecified")
total_cases = len(filtered_df)

# Create the bar chart with percentage calculation
age_chart = alt.Chart(filtered_df).mark_bar().encode(
    x=alt.X("Age Category:N", sort="-y", title="Age Group", axis=alt.Axis(labelAngle=45)),
    y=alt.Y("Percentage:Q", title="Proportion (%)"),
    color=alt.Color("Age Category:N", legend=None),
    tooltip=["Age Category:N", "Percentage:Q"]
).transform_aggregate(
    count_cases="count()",
    groupby=["Age Category"]
).transform_calculate(
    Percentage="100 * datum.count_cases / {}".format(total_cases)
).properties(
    width=600,
    height=400
)



# age_chart.save("./graphs/age group percentage.pdf")
age_chart

In [ ]:
def extract_age(strings):
 
    results = []

    # Regex pattern to extract age with units (e.g., "26-year-old", "2 months old")
    age_pattern = re.compile(
        r'(\b(?:\d+|(?:one|two|three|four|five|six|seven|eight|nine|ten|eleven|twelve|thirteen|fourteen|fifteen|sixteen|seventeen|eighteen|nineteen|twenty|thirty|forty|fifty|sixty|seventy|eighty|ninety)(?:[-\s]?(?:one|two|three|four|five|six|seven|eight|nine))?)\b)[-\s]*(year[s]?|yr[s]?|month[s]?|mon|y[s]?|wk[s]?|w[s]?|week[s]?|d[s]?|day[s]?)[-\s]*(old|of age)?', 
        re.IGNORECASE
    )


    for text in strings:
        # Search for age information
        match_age = age_pattern.search(text)
        if match_age:
            age_text = match_age.group(1)  # Extract the numeric or word-based part
            age_unit = match_age.group(2).lower()  # Extract the unit part
            
            # Convert word-based numbers to integer if necessary
            try:
                age_value = int(age_text) if age_text.isdigit() else w2n.word_to_num(age_text)
            except ValueError:
                age_value = None
            
            # Convert to years if age is in months, weeks, or days
            if age_value is not None:
                if 'month' in age_unit or 'mon' in age_unit:
                    age_in_years = round(age_value / 12, 2)
                    age = f"{age_in_years} years"
                elif 'week' in age_unit:
                    age_in_years = round(age_value / 52, 2)
                    age = f"{age_in_years} years"
                elif 'day' in age_unit:
                    age_in_years = round(age_value / 365, 2)
                    age = f"{age_in_years} years"
                else:
                    # If the age is already in years
                    age = f"{age_value} years"
            else:
                age = "unspecified"
        else:
            age = "unspecified"

        # Store the extracted age
        results.append({'Age': age})
    
    return results

In [ ]:
age = extract_age(dataset["case"])
age_dict = {case: i["Age"] for case, i in zip(dataset["pmcid"], age)}
age_df = pd.DataFrame(list(age_dict.items()), columns=["pmcid", "Age"])

In [ ]:
len({case: i["Age"] for case, i in zip(dataset["pmcid"], age)if i["Age"]=="unspecified"})/len(dataset["case"])

In [ ]:
pmcid_counts = Counter(dataset["pmcid"])
duplicates = {k: v for k, v in pmcid_counts.items() if v > 1}
print(f"Total duplicate pmcids: {len(duplicates)}")

In [ ]:
age_df["Age (years)"] = age_df["Age"].str.extract(r'(\d+\.?\d*)')  # Now captures decimals too
# Convert only valid numbers to float, leave NaN as is
age_df["Age (years)"] = pd.to_numeric(age_df["Age (years)"], errors='coerce')

bins = [0, 1/12, 1.5, 11, 16, 41, 65, float('inf')]  # Updated bins for age groups
labels = [
    'Neonatal',    # 0 - 1 month
    'Infancy',     # >1 month - 1.5 years
    'Childhood',   # >1.5 - 11 years
    'Adolescence', # >11 - 16 years
    'Adulthood (16-41 yr)',  # >16 - 41 years
    'Adulthood (41-65 yr)',  # >41 - 65 years
    'Adulthood (>65 yr)'     # >65 years
]


# Bin numeric ages
age_df["Age Category"] = pd.cut(pd.to_numeric(age_df["Age (years)"], errors='coerce'), bins=bins, labels=labels, right=True)
# Convert all NaN values in "Age Category" to "Unspecified"
age_df["Age Category"] = age_df["Age Category"].astype(str).replace({"nan": "Unspecified", np.nan: "Unspecified"})

# Convert to string type before assigning "Unspecified"
age_df["Age Category"] = age_df["Age Category"].astype(str)

# Assign "Unspecified" for rows where Age was missing
age_df.loc[age_df["Age"].str.lower().str.contains("unspecified", na=False), "Age Category"] = "Unspecified"

In [ ]:
print(age_df[age_df["Age (years)"].isna()]["Age"].unique())
print(age_df["Age Category"].unique())  

In [ ]:
iltered_df = age_df[age_df["Age Category"].notna()]

# Compute total number of cases (excluding "Unspecified")
total_cases = len(filtered_df)

# Create the bar chart with percentage calculation
age_chart = alt.Chart(filtered_df).mark_bar().encode(
    x=alt.X("Age Category:N", sort="-y", title="Age Group", axis=alt.Axis(labelAngle=45)),
    y=alt.Y("Percentage:Q", title="Proportion (%)"),
    color=alt.Color("Age Category:N", legend=None),
    tooltip=["Age Category:N", "Percentage:Q"]
).transform_aggregate(
    count_cases="count()",
    groupby=["Age Category"]
).transform_calculate(
    Percentage="100 * datum.count_cases / {}".format(total_cases)
).properties(
    width=600,
    height=400
)



# age_chart.save("./graphs/age group percentage.pdf")
age_chart

##  Biological Sex

In [ ]:
with open("./data/merged_extracted_sex.json") as f:
    sex = json.load(f)
    
sex_df = pd.DataFrame(list(sex.items()), columns=["pmcid", "sex"])

sex_df["Sex"] = sex_df["sex"].replace({None: "unspecified"}) # convert failed to extraction to "unspecified" 
sex_df["pmcid"] = sex_df["pmcid"].astype(str)

In [ ]:
sex_age_granular_df =  age_df[["Age", "Age Category", "pmcid"]].merge(sex_df[["Sex", "pmcid"]]) 

In [ ]:
merged_df = yr_df.merge(age_df[["Age Category", "pmcid"]]).merge(sex_df[["Sex", "pmcid"]]).merge(df[["combined_label_2","pmcid"]])

In [ ]:
merged_df = merged_df.rename(columns={"Age Category": "age", "Sex": "sex", "combined_label_2": "topic"}) 

In [ ]:
case_df =  category_df(process_extraction())

In [ ]:
finalized_df = merged_df.merge(case_df[['pmcid', 'title', 'case', 'case_length', 'Vitals_Hema', 'Pregnancy',
       'Neuro', 'CVS', 'RESP', 'EENT', 'GI', 'GU', 'DERM', 'MSK', 'ENDO',
       'LYMPH', 'History', 'Lab_Image']])

* Remove erroneously extracted pregnancy information for males if any. There should be none by now

In [ ]:
finalized_df.loc[finalized_df["sex"] == "male", "Pregnancy"] = finalized_df.loc[finalized_df["sex"] == "male", "Pregnancy"].apply(lambda x: [])

* sex distribution per publication yr


In [ ]:
import pandas as pd
import altair as alt

# Ensure 'publication_year' is integer and remove NaNs
finalized_df["publication_year"] = pd.to_numeric(finalized_df["publication_year"], errors="coerce").fillna(0).astype(int)

# Drop rows with missing labels or invalid years
df_filtered = finalized_df.dropna(subset=["sex", "publication_year"])
df_filtered = df_filtered[(df_filtered["publication_year"] >0) & (df_filtered["publication_year"] < 2024)]

# Count occurrences of each (Year, Sex) pair
label_freq = df_filtered.groupby(["publication_year", "sex"]).size().reset_index(name="Count")
label_freq.rename(columns={"publication_year": "Year"}, inplace=True)

 
# Create stacked bar chart
chart = alt.Chart(label_freq).mark_bar().encode(
    x=alt.X("Year:O", title="Publication Year",  axis=alt.Axis(labelAngle=-90, titleFontSize=30, labelFontSize=30)),
    y=alt.Y("Count:Q", title="Case Counts", stack="zero",  axis=alt.Axis(labelAngle=0, titleFontSize=30, labelFontSize=30)),
    color=alt.Color("sex:N", scale=color_scheme, legend=alt.Legend(title="Sex", titleFontSize=30, labelFontSize=30)),  # Fixed incorrect 'Label' reference
    tooltip=["Year", "sex", "Count"]
).properties(
    width=1000,
    height=300,
)

chart.save("./graphs/sex distribution over pub yrs.pdf")
# Display the chart
chart


* Sex distribution per conditions

In [ ]:
df_expanded = finalized_df.copy()
df_expanded["topic"] = df_expanded["topic"].apply(
    lambda x: [re.sub(r"\s+", " ", topic).strip() for topic in x.split(',')] if isinstance(x, str) else x
)
df_expanded = df_expanded.explode("topic")

In [ ]:
df_expanded["topic"].value_counts()[:20]

In [ ]:
 
df_expanded = df_expanded.explode("topic").dropna(subset=["topic", "sex"])

# Get the top 20 most frequent topics **by name**
top_20_topics = df_expanded["topic"].value_counts().nlargest(20).index.tolist()

# Filter for only these top 20 topics
df_filtered = df_expanded[df_expanded["topic"].isin(top_20_topics)]

# Count occurrences of each (Topic, Sex) pair
topic_sex_counts = df_filtered.groupby(["topic", "sex"]).size().reset_index(name="Count")

# Create stacked bar chart showing **raw counts** of sex per topic
chart = alt.Chart(topic_sex_counts).mark_bar().encode(
    x=alt.X("topic:N", title=None, sort="-y", axis=alt.Axis(labelAngle=45, titleFontSize=30, labelFontSize=30, labelLimit=500)),
    y=alt.Y("Count:Q", title="Case Counts", stack="zero", axis=alt.Axis(titleFontSize=30, labelFontSize=30)),  
    color=alt.Color("sex:N", legend=alt.Legend(title="Sex", titleFontSize=30, labelFontSize=25),scale=color_scheme),
    tooltip=["topic", "sex", "Count"]
).properties(
    width=1000,
    height=300
)

chart.save("./graphs/sex distribution over top 20 medical conditions.pdf")
# Display the chart
chart


* sex  per age group

In [ ]:
# Count occurrences of each (Year, Sex) pair
custom_order = [
    "Neonatal", "Infancy", "Childhood", "Adolescence",
    "Adulthood (16-41 yr)", "Adulthood (41-65 yr)", 
    "Adulthood (>65 yr)", "Unspecified"
]
color_scheme = alt.Scale(scheme='sinebow') 
label_freq = finalized_df.groupby(["age", "sex"]).size().reset_index(name="Count")

total_counts = label_freq.groupby("age")["Count"].sum().reset_index()

# sorted_ages = total_counts.sort_values(by="Count", ascending=False)["age"].tolist()

# Create stacked bar chart (stacked bars for sex per year)
chart = alt.Chart(label_freq).mark_bar().encode(
    x=alt.X("age:O", title="Age Group", sort=custom_order, axis=alt.Axis(labelAngle=-90, titleFontSize=30, labelFontSize=30)),
    y=alt.Y("Count:Q", title="Case Counts", stack="zero",  axis=alt.Axis(titleFontSize=30, labelFontSize=30)),   
    color=alt.Color("sex:N", legend=alt.Legend(title="Sex", titleFontSize=30, labelFontSize=25), scale=color_scheme),   
    tooltip=["age", "sex", "Count"]
).properties(
    width=1000,
    height=300
)
chart.save("./graphs/sex distribution over age groups.pdf")
# Display the chart
chart


# Sex Composition in Dataset

In [ ]:
sex_df["Sex"].value_counts(normalize=True).apply(lambda x: round(x*100, 2) )

# Chi-square test

* Is age composition remain stable across different publication years

In [ ]:
# Filter out rows where 'publication_year' is "Unknown Year"
df_filtered = merged_df[(merged_df["publication_year"] != "Unknown Year") & (merged_df["sex"] != "unspecified") & (merged_df["age"] != "Unspecified") ]


In [ ]:
# Create contingency table
contingency_table = pd.crosstab(df_filtered["publication_year"], df_filtered["age"])

# Perform Chi-Square test
chi2, p, dof, expected = stats.chi2_contingency(contingency_table)

# Display results
print("Chi-Square Statistic:", chi2)
print("p-value:", p)
print("Degrees of Freedom:", dof)

# Check significance
if p < 0.05:
    print("Significant difference in age distribution across publication years.")
else:
    print("No significant difference in age distribution across publication years.")


In [ ]:
contingency_table

In [ ]:
past_10_yrs = df_filtered[df_filtered["publication_year"].astype(int)<=2012]
# Create contingency table
contingency_table = pd.crosstab(past_10_yrs["publication_year"], past_10_yrs["age"])

# Perform Chi-Square test
chi2, p, dof, expected = stats.chi2_contingency(contingency_table)

# Display results
print("Chi-Square Statistic:", chi2)
print("p-value:", p)
print("Degrees of Freedom:", dof)

# Check significance
if p < 0.05:
    print("Significant difference in age distribution across publication years between 2013 and 2023.")
else:
    print("No significant difference in age distribution across publication years between 2013 and 2023.")


* Is sex composition remain stable across different publication years

In [ ]:
# Create contingency table
contingency_table = pd.crosstab(df_filtered["publication_year"], df_filtered["sex"])

# Perform Chi-Square test
chi2, p, dof, expected = stats.chi2_contingency(contingency_table)

# Display results
print("Chi-Square Statistic:", chi2)
print("p-value:", p)
print("Degrees of Freedom:", dof)

# Check significance
if p < 0.05:
    print("Significant difference in sex distribution across publication years.")
else:
    print("No significant difference in sex distribution across publication years.")


In [ ]:
contingency_table

* Is sex composition remain stable across age groups?

In [ ]:
 # Create contingency table
contingency_table = pd.crosstab(df_filtered["age"], df_filtered["sex"])

# Perform Chi-Square test
chi2, p, dof, expected = stats.chi2_contingency(contingency_table)

# Display results
print("Chi-Square Statistic:", chi2)
print("p-value:", p)
print("Degrees of Freedom:", dof)

# Check significance
if p < 0.05:
    print("Significant difference in sex distribution across age groups.")
else:
    print("No significant difference in sex distribution across age groups.")


* Is sex composition remain stable across different medical topics? 

In [ ]:
 # Create contingency table
contingency_table = pd.crosstab(df_filtered["topic"], df_filtered["sex"])

# Perform Chi-Square test
chi2, p, dof, expected = stats.chi2_contingency(contingency_table)

# Display results
print("Chi-Square Statistic:", chi2)
print("p-value:", p)
print("Degrees of Freedom:", dof)

# Check significance
if p < 0.05:
    print("Significant difference in sex distribution across medical topics.")
else:
    print("No significant difference in sex distribution across medical topics.")



 * For medical topics with at least 100 case reports:

In [ ]:
df_top_100 = df_filtered[df_filtered["topic"].isin(topics_with_at_least_100)]

In [ ]:
 # Create contingency table
contingency_table = pd.crosstab(df_top_100["topic"], df_top_100["sex"])

# Perform Chi-Square test
chi2, p, dof, expected = stats.chi2_contingency(contingency_table)

# Display results
print("Chi-Square Statistic:", chi2)
print("p-value:", p)
print("Degrees of Freedom:", dof)
        
# Check significance
if p < 0.05:
    print("Significant difference in sex distribution across medical topics.")
else:
    print("No significant difference in sex distribution across medical topics.")



In [ ]:
contingency_table

In [ ]:
merged_df

In [ ]:
# Filter out unknown years
df_filtered = merged_df[merged_df["publication_year"] != "Unknown Year"]

# Create a contingency table of age-sex distribution per year
contingency_table = pd.crosstab(index=df_filtered["publication_year"], columns=[df_filtered["age"], df_filtered["sex"]])

# Perform Chi-Square test
chi2, p, dof, expected = chi2_contingency(contingency_table)

# Print results
print(f"Chi-Square Statistic: {chi2}")
print(f"p-value: {p}")
print(f"degree_of_freedom: {dof}")

# Check significance
if p < 0.05:
    print("Significant difference in age-sex distribution across years.")
else:
    print("No significant difference in age-sex distribution across years.")


In [ ]:
df_filtered["age"].unique()

In [ ]:
# Convert to lowercase and remove spaces
merged_df["age"] = merged_df["age"].str.lower().str.strip()
merged_df["sex"] = merged_df["sex"].str.lower().str.strip()

# Now filter correctly
df_filtered = merged_df[(merged_df["age"] != "unspecified") & (merged_df["sex"] != "unspecified")]


# Create a contingency table of age-sex distribution per year
contingency_table = pd.crosstab(index=df_filtered ["topic"], columns=[df_filtered ["age"], df_filtered ["sex"]])

# Perform Chi-Square test
chi2, p, dof, expected = chi2_contingency(contingency_table)

# Print results
print(f"Chi-Square Statistic: {chi2}")
print(f"p-value: {p}")
print(f"degree_of_freedom: {dof}")
# Check significance
if p < 0.05:
    print("Significant difference in age-sex distribution across topics.")
else:
    print("No significant difference in age-sex distribution across topics.")


In [ ]:
# import altair as alt
# from scipy.stats import zscore
# import numpy as np

# # Compute Z-score for residuals safely
# if residuals_long["residual_value"].std() != 0:
#     residuals_long["z_score"] = zscore(residuals_long["residual_value"])
# else:
#     residuals_long["z_score"] = 0

# # Clip extreme values
# residuals_long["z_score"] = residuals_long["z_score"].clip(lower=-5, upper=5)

# # Filter only statistically significant residuals
# filtered_residuals = residuals_long[(residuals_long["z_score"] > 2) | (residuals_long["z_score"] < -2)]

# # Create heatmap for filtered residuals with adjusted color scale
# heatmap = alt.Chart(filtered_residuals).mark_rect().encode(
#     x=alt.X("age_sex:N", title="Age-Sex Group", axis=alt.Axis(labelAngle=-45)),
#     y=alt.Y("topic_:N", title=None, sort="-x", axis=alt.Axis(labels=False, ticks=False)),
#     color=alt.Color("z_score:Q", title="Z-Score (Normalized Residuals)", 
#                     scale=alt.Scale(scheme="redblue", domain=[-5, 5])),  # Fix color balance
#     tooltip=["topic_", "age_sex", "z_score"]
# ).properties(
#     width=900,
#     height=600,
#     title="Heatmap of Statistically Significant Over-/Underrepresented Topics by Age-Sex Group"
# )

# heatmap.show()


In [ ]:

# # Perform Chi-Square test
# chi2, p, dof, expected = chi2_contingency(contingency_table)

# # Compute standardized residuals
# residuals = (contingency_table - expected) / np.sqrt(expected)
# residuals = residuals.reset_index()
 
# # Flatten the MultiIndex column names

# residuals.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in residuals.columns]

# # Convert residuals table to long format for sortings
# residuals_long = residuals.melt(id_vars=["topic_"], var_name="age_sex", value_name="residual_value")



* Which age-sex combination is over or under represented in each topic


In [ ]:
# top_10_overrepresented = residuals_long.sort_values(by="residual_value", ascending=False).head(10)
# top_10_overrepresented

In [ ]:
# top_10_underrepresented = residuals_long.sort_values(by="residual_value", ascending=True).head(10)
# top_10_underrepresented


In [ ]:
finalized_df

In [ ]:
hf_dataset = Dataset.from_pandas(finalized_df)